## Parameter Settings

In [1]:
DATA_ROOT_DIR = '/workspace/data'
VIDEOS_DIR = f'{DATA_ROOT_DIR}/videos'
annotation_path = f'{DATA_ROOT_DIR}/annotations.jsonl'

In [2]:
# read annotation file as pandas dataframe
import json
import pandas as pd

with open(annotation_path, 'r') as f:
    annotations = [json.loads(line) for line in f]

df_annotations = pd.DataFrame(annotations) 

# 折り返さずに表示する設定
pd.set_option('display.max_colwidth', 200)

print(df_annotations.head())

                           video_path           selected_class  \
0            wyzi9GNZFMU_0_0to121.mp4    Camera Motion Editing   
1  8rKYl1CdXCc_5_276to660_scene02.mp4  Instance Motion Editing   
2           1s9DER1bpm0_10_0to213.mp4         Quantity Editing   
3            DaUJkmBvTKM_2_0to150.mp4    Visual Effect Editing   
4            Xw9Zsc9A924_0_0to138.mp4        Attribute Editing   

   selected_subclass  \
0           Dolly in   
1       Human motion   
2           Increase   
3  Background Change   
4   Color adjustment   

                                                                                                                                                                                               instruction  
0                                                        Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.  
1  Modify the video so that the man raises his l

In [3]:
# df の selected_class のユニークな値を取得する
unique_classes = df_annotations['selected_class'].unique()
print(unique_classes)

['Camera Motion Editing' 'Instance Motion Editing' 'Quantity Editing'
 'Visual Effect Editing' 'Attribute Editing' 'Style Editing'
 'Instance Editing' 'Camera Angle Editing']


In [4]:
# df の selected_subclass のユニークな値を取得する
unique_subclasses = df_annotations['selected_subclass'].unique()
print(unique_subclasses)

['Dolly in' 'Human motion' 'Increase' 'Background Change'
 'Color adjustment' 'Ukiyo-e' 'Zoom in' 'Instance Replacement' 'Cyberpunk'
 'Decoration effect' 'Instance Insertion' 'Pixel' 'Instance Removal'
 'Zoom out' 'Low angle' 'Anime' 'object motion' 'High angle' 'Ghibli'
 'Watercolor' 'Oil painting' 'American comic style' 'Arc shot']


In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json

# ==========================================
# モデルロード
# ==========================================
MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# 追加（必須）
# MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"　は　eos_token を定義していないため、
# tokenizer.pad_token に eos_token を設定する必要がある
tokenizer.pad_token = tokenizer.eos_token

# 追加-2
tokenizer.padding_side = "left"   # ★これが必須

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()  # 背景意図: dropout等を無効化し、推論を安定＆高速化

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 291/291 [00:03<00:00, 78.37it/s]


MistralForCausalLM(
  (model): MistralModel(
    (embed_tokens): Embedding(32000, 4096)
    (layers): ModuleList(
      (0-31): 32 x MistralDecoderLayer(
        (self_attn): MistralAttention(
          (q_proj): Linear(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (v_proj): Linear(in_features=4096, out_features=1024, bias=False)
          (o_proj): Linear(in_features=4096, out_features=4096, bias=False)
        )
        (mlp): MistralMLP(
          (gate_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (up_proj): Linear(in_features=4096, out_features=14336, bias=False)
          (down_proj): Linear(in_features=14336, out_features=4096, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): MistralRMSNorm((4096,), eps=1e-05)
        (post_attention_layernorm): MistralRMSNorm((4096,), eps=1e-05)
      )
    )
    (norm): MistralRMSNorm((4096,)

In [6]:
# ==========================================
# プロンプト
# ==========================================
def build_prompt(instruction):
    """
    改善ポイント:
    - 出力制御を強化（Explanationを防ぐ）
    - schema逸脱を防ぐ
    - token数削減（速度改善）
    """

    return f"""
You are a video editing parser.

Return ONLY JSON. No explanation. No extra text.

Allowed values:
- group: camera | other
- task: zoom_in | zoom_out | low_angle | high_angle | arc_shot | unknown
- target_kind: face | person | object | scene | unknown
- motion_style: smooth | gradual | steady | slow | none

If uncertain, use "unknown".

Instruction:
{instruction}

JSON:
"""


# ==========================================
# 推論
# ==========================================
def parse_with_llm(instruction):

    prompt = build_prompt(instruction)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # 背景意図:
    # - 毎回 to(device) を呼ぶとオーバーヘッドが出るため、
    #   dict内包で一括転送
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,      # 背景意図: JSONのみ生成に制限（無駄生成削減）
            do_sample=False,         # 背景意図: deterministic化 → JSON崩壊防止
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ==========================================
    # JSON抽出（強化版）
    # ==========================================
    try:
        json_str = text.split("JSON:")[-1].strip()

        # 背景意図:
        # - 余計な文章が付いた場合でも最初のJSONだけ抜く
        start = json_str.find("{")
        end = json_str.rfind("}") + 1
        json_str = json_str[start:end]

        parsed = json.loads(json_str)

    except Exception as e:
        parsed = {"error": text}

    return parsed

## 高速化
ver_01　が　42分/100件　かかったので、高速化を検討

In [ ]:
# ==========================================
# 実行（batch化）
# ==========================================
instructions = df_annotations['instruction'].tolist()

# print　-> .txtに出力する
output_file = "parsed_instructions_ver02_1.txt"

num_total = len(instructions)
num_skipped = 0

BATCH_SIZE = 8  # 背景意図: GPU並列活用（精度に影響なし）--> VRAM=23.1[GB]

for i in range(0, num_total, BATCH_SIZE):

    batch = instructions[i:i+BATCH_SIZE]

    prompts = [build_prompt(inst) for inst in batch]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for inst, text in zip(batch, texts):

        try:
            json_str = text.split("JSON:")[-1].strip()
            start = json_str.find("{")
            end = json_str.rfind("}") + 1
            json_str = json_str[start:end]

            parsed = json.loads(json_str)

        except:
            parsed = {"error": text}
            num_skipped += 1

        print(f"Instruction: {inst}")
        print(f"Parsed: {parsed}")
        print("-" * 50)

        # 出力をファイルに書き込む
        with open(output_file, "a", encoding="utf-8") as f:
            f.write(f"Instruction: {inst}\n")
            f.write(f"Parsed: {parsed}\n")
            f.write("-" * 50 + "\n")


print(f"Total instructions: {num_total}")
print(f"Skipped instructions: {num_skipped}")

# printを.txt に出力する
with open(output_file, "a", encoding="utf-8") as f:
    f.write(f"Total instructions: {num_total}\n")
    f.write(f"Skipped instructions: {num_skipped}\n")

Instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
Parsed: {'group': 'camera', 'task': 'zoom_in', 'target_kind': 'face', 'motion_style': 'smooth'}
--------------------------------------------------
Instruction: Modify the video so that the man raises his left hand to wave towards the camera in a friendly manner. Preserve his identity, the suit he is wearing, and the dark blue stage background. Ensure the motion is natural and temporally consistent, with no flickering or artifacts around the hand and arm.
Parsed: {'error': '\nYou are a video editing parser.\n\nReturn ONLY JSON. No explanation. No extra text.\n\nAllowed values:\n- group: camera | other\n- task: zoom_in | zoom_out | low_angle | high_angle | arc_shot | unknown\n- target_kind: face | person | object | scene | unknown\n- motion_style: smooth | gradual | steady | slow | none\n\nIf uncertain, use "unknown".\n\nInstructio

# 結果 -\> 精度低下

## 結論（数値ベース）

- 総件数：108件\

- 正常JSON：34件

- エラー：74件

------------------------------------------------------------------------

## ① 改善はしているか？

### 速度

42分 → 50秒\
約 50倍高速化

→ batch化は成功

### 精度

|状態|     件数|   割合|
|--------| ------| -------|
|正常     |34     |31.5%|
|エラー   |74     |68.5%|

------------------------------------------------------------------------

## ② 前回との比較（重要）

### before（single推論）

- 約100件中 → ほぼ成功（体感）\
- 1件25秒

### after（batch + left pad）

- 成功率：31%\
- 速度：爆速

------------------------------------------------------------------------

## ③ 問題の本質（根拠あり）

エラー内容を見ると：

motion: \[ { "group": "camera", ...

→ JSONではなく「構造生成」に逸脱

### パターン分析

#### エラーの特徴（74件すべて共通）

- JSONではなく途中で崩壊\
- schema外のキー（motion, camera, duration など）\
- 複数JSON or 配列生成\
- プロンプト再出力（instruction繰り返し）

------------------------------------------------------------------------

## ④ なぜこうなったか（原因）

### 原因①：batchで制約が弱まる

#### single時：

1 instruction → 1文脈

#### batch時：

複数instruction → 文脈が混ざる

→ LLMが「生成タスク」に逃げる

------------------------------------------------------------------------

### 原因②：プロンプト拘束が弱い

現在：

Return ONLY JSON

→ 弱すぎる

------------------------------------------------------------------------

### 原因③：max_new_tokens過剰

現在：

max_new_tokens=120

→ 長すぎて「余計な生成」に逸脱

------------------------------------------------------------------------

## ⑤ 状態評価

|項目           |評価|
|-------------- |-------------------------|
|left padding   |✅ 効いてる（速度改善）|
|batch処理      |⚠️ 部分成功|
|JSON精度       |❌ 不合格|

------------------------------------------------------------------------

## ⑥ 改善しないとダメなポイント（優先度順）

① プロンプト強制（最重要）

今のままだと100%安定しない

② max_new_tokens削減\
③ JSON抽出方法が弱い

text.split("JSON:")\[-1\]

→ 壊れる原因

------------------------------------------------------------------------

## ⑦ 次にやるべき修正（最短ルート）

### 修正①（必須）
```python
max_new_tokens=60
```
### 修正②（プロンプト強化）
```python
Return ONLY a single JSON object.

Do NOT output: - arrays - multiple JSON objects - explanations - any
text before or after JSON

The output MUST: - start with '{' - end with '}' - contain all required
keys

If unsure, use "unknown".
```
------------------------------------------------------------------------

### 修正③（JSON抽出改善）
```python
import re

def extract_json(text): match = re.search(r"{.\*}", text, re.DOTALL) if
match: return json.loads(match.group()) else: return {"error": text}
```
------------------------------------------------------------------------

## ⑧ 重要な判断

### 今の設計の限界

LLMに「自由生成」させている

→ だから壊れる


## 改善
max_new_tokens=60

In [8]:
# ==========================================
# プロンプト
# ==========================================
def build_prompt(instruction):
    """
    改善ポイント:
    - 出力制御を強化（Explanationを防ぐ）
    - schema逸脱を防ぐ
    - token数削減（速度改善）
    """

    return f"""
You are a video editing parser.

Return ONLY JSON. No explanation. No extra text.

Allowed values:
- group: camera | other
- task: zoom_in | zoom_out | low_angle | high_angle | arc_shot | unknown
- target_kind: face | person | object | scene | unknown
- motion_style: smooth | gradual | steady | slow | none

If uncertain, use "unknown".

Instruction:
{instruction}

JSON:
"""


# ==========================================
# 推論
# ==========================================
def parse_with_llm(instruction):

    prompt = build_prompt(instruction)

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    )

    # 背景意図:
    # - 毎回 to(device) を呼ぶとオーバーヘッドが出るため、
    #   dict内包で一括転送
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=60,      # 背景意図: JSONのみ生成に制限（無駄生成削減）
            do_sample=False,         # 背景意図: deterministic化 → JSON崩壊防止
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # ==========================================
    # JSON抽出（強化版）
    # ==========================================
    try:
        json_str = text.split("JSON:")[-1].strip()

        # 背景意図:
        # - 余計な文章が付いた場合でも最初のJSONだけ抜く
        start = json_str.find("{")
        end = json_str.rfind("}") + 1
        json_str = json_str[start:end]

        parsed = json.loads(json_str)

    except Exception as e:
        parsed = {"error": text}

    return parsed

In [9]:
# ==========================================
# 実行（batch化）
# ==========================================
instructions = df_annotations['instruction'].tolist()

# print　-> .txtに出力する
output_file = "parsed_instructions_ver02_2.txt"

num_total = len(instructions)
num_skipped = 0

BATCH_SIZE = 8  # 背景意図: GPU並列活用（精度に影響なし）--> VRAM=23.1[GB]

for i in range(0, num_total, BATCH_SIZE):

    batch = instructions[i:i+BATCH_SIZE]

    prompts = [build_prompt(inst) for inst in batch]

    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=120,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.eos_token_id,
        )

    texts = tokenizer.batch_decode(outputs, skip_special_tokens=True)

    for inst, text in zip(batch, texts):

        try:
            json_str = text.split("JSON:")[-1].strip()
            start = json_str.find("{")
            end = json_str.rfind("}") + 1
            json_str = json_str[start:end]

            parsed = json.loads(json_str)

        except:
            parsed = {"error": text}
            num_skipped += 1

        print(f"Instruction: {inst}")
        print(f"Parsed: {parsed}")
        print("-" * 50)

        # 出力をファイルに書き込む
        with open(output_file, "a", encoding="utf-8") as f:
            f.write(f"Instruction: {inst}\n")
            f.write(f"Parsed: {parsed}\n")
            f.write("-" * 50 + "\n")


print(f"Total instructions: {num_total}")
print(f"Skipped instructions: {num_skipped}")

# printを.txt に出力する
with open(output_file, "a", encoding="utf-8") as f:
    f.write(f"Total instructions: {num_total}\n")
    f.write(f"Skipped instructions: {num_skipped}\n")

Instruction: Apply a smooth dolly in effect toward the man's face, starting from the original medium shot and ending in a close-up while keeping him centered.
Parsed: {'group': 'camera', 'task': 'zoom_in', 'target_kind': 'face', 'motion_style': 'smooth'}
--------------------------------------------------
Instruction: Modify the video so that the man raises his left hand to wave towards the camera in a friendly manner. Preserve his identity, the suit he is wearing, and the dark blue stage background. Ensure the motion is natural and temporally consistent, with no flickering or artifacts around the hand and arm.
Parsed: {'error': '\nYou are a video editing parser.\n\nReturn ONLY JSON. No explanation. No extra text.\n\nAllowed values:\n- group: camera | other\n- task: zoom_in | zoom_out | low_angle | high_angle | arc_shot | unknown\n- target_kind: face | person | object | scene | unknown\n- motion_style: smooth | gradual | steady | slow | none\n\nIf uncertain, use "unknown".\n\nInstructio

# まとめ

60　への変更だけ反映。\
結果は添付の通りで、変化なし

まずは、プロンプトを変えずに、高速化して、同じ結果が出るようにしたい。\
なぜ結果が変わる？

------------------------------------------------------------------------

結論から述べる。

batch化により「入力条件」が変わっているため、\
同じプロンプトでも同じ結果は出ない

これはバグではなく LLMの仕様。

------------------------------------------------------------------------

## ① 何が変わったか（根拠ベース）

### single推論（元の状態）

    [ prompt tokens ]
    → generate

### batch推論（現在）

    [ PAD PAD PAD prompt A ]
    [ PAD PAD prompt B ]
    [ PAD prompt C ]

    → これをまとめて forward

------------------------------------------------------------------------

## ② 本質的な違い

### 違い①：attention構造

LLM内部では：

各バッチは独立\
だが長さが揃えられる

問題\
「どのtokenが最後か」が変わる

------------------------------------------------------------------------

### 違い②：生成開始位置

#### single：

最後のtoken → 正しい文脈

#### batch：

paddingの影響で位置ズレ

------------------------------------------------------------------------

### 違い③：KV cacheの挙動

batchは内部で並列処理される\
→ 微妙な数値差が発生

------------------------------------------------------------------------

## ③ なぜ結果が変わるのか（核心）

### 原因1：paddingによる位置ズレ

あなたは修正した：

    tokenizer.padding_side = "left"

→ これは正しい

しかし：

「完全一致」にはならない

理由：

position encoding が変わる

------------------------------------------------------------------------

### 原因2：生成の非決定性

あなたは：

    temperature=0.0

だが：

完全決定ではない

理由：

-   float16誤差\
-   GPU並列計算\
-   fused kernel

------------------------------------------------------------------------

### 原因3：batchの長さ依存

一番長いpromptに合わせて全体が引っ張られる

→ 短いpromptほど影響を受ける

------------------------------------------------------------------------

## ④ 実験結果からの裏付け

あなたのログ：

-   成功率：約31%\
-   エラー：74件

### 特徴

構造崩壊（motion: 〜）

→ これは：

文脈ズレの典型パターン

------------------------------------------------------------------------

## ⑤ 「同じ結果にしたい」場合の条件

### 必要条件（厳密）

1.  paddingなし\
2.  batchサイズ=1\
3.  deterministic演算

→ 結論：

batchと完全一致は不可能

------------------------------------------------------------------------

## ⑥ ではどうするか（現実解）

### 方法①：長さ揃え

    max_length = 固定値

→ 効果：

position差減少

------------------------------------------------------------------------

### 方法②：length別batch

短文だけのbatch\
長文だけのbatch

→ 効果：

padding影響減

------------------------------------------------------------------------

### 方法③：逐次generate（重要）

``` python
for i in batch:
    model.generate()
```

→ 内部はsingleと同じ

------------------------------------------------------------------------

## ⑦ あなたの目的に対する最適解

あなたの要求：

・高速化\
・同じ結果

### 正解

tokenizeはbatch\
generateはsingle

### 実装

``` python
inputs = tokenizer(prompts, ...)

for i in range(len(prompts)):
    single_input = {
        k: v[i:i+1].to(model.device)
        for k, v in inputs.items()
    }

    output = model.generate(**single_input)
```

------------------------------------------------------------------------

## ⑧ なぜこれが最適か

  方法                               |速度   |再現性|
|---------------------------------- |------ |--------|
|完全batch                          |◎      |❌|
|完全single                         |❌     |◎|
|tokenize batch + generate single   |○      |◎|

------------------------------------------------------------------------

## ⑨ 重要な理解

LLMは「関数」ではない

→ 入力が同じでも結果が揺れる

------------------------------------------------------------------------

## ⑩ 一言まとめ

batch化で壊れているのではなく\
「別の問題を解いている状態」になっている
